In [1]:
# Tasks 1 to 4: Data collection from Binance, saving JSON, and running analytical summaries
import requests
import json

def fetch_and_analyze_crypto():
    url = "https://api.binance.com/api/v3/ticker/24hr"
    
    # Defining a specific scope of the top 10 highly liquid markets to isolate
    target_symbols = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "ADAUSDT", 
                      "XRPUSDT", "DOTUSDT", "DOGEUSDT", "AVAXUSDT", "LINKUSDT"]
    
    try:
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            all_tickers = response.json()
            
            # Task 1: Filter down the dataset to our 10 target symbols
            filtered_data = [ticker for ticker in all_tickers if ticker['symbol'] in target_symbols]
            
            # Persist raw tracking data into a local JSON archive
            with open("crypto_data.json", mode="w", encoding="utf-8") as json_file:
                json.dump(filtered_data, json_file, indent=4)
            print("Successfully archived real-time data array matrix to 'crypto_data.json'.\n")
            
            # Trigger custom analytical steps
            run_analytics_dashboard(filtered_data)
            
        elif response.status_code == 429:
            print("[Rate Limit Warning] Received HTTP 429. Backing off.")
            return response.status_code
        else:
            print(f"Failed connection sequence. Status Code: {response.status_code}")
            
    except requests.exceptions.RequestException as e:
        print(f"Network error encountered: {e}")
    return None

# Task 2: Find the coin with the absolute largest 24-hour price change percentage
def find_most_volatile_coin(data):
    highest_volatility = -1.0
    volatile_symbol = ""
    
    for coin in data:
        # Binance returns parameters as string types; casting to float is necessary
        price_change_pct = abs(float(coin.get("priceChangePercent", 0.0)))
        if price_change_pct > highest_volatility:
            highest_volatility = price_change_pct
            volatile_symbol = coin.get("symbol")
            
    return volatile_symbol, highest_volatility

# Task 3: Calculate the mean valuation and flag values trailing that baseline
def display_coins_below_average(data):
    total_price = sum(float(coin.get("lastPrice", 0.0)) for coin in data)
    avg_price = total_price / len(data)
    
    print(f"Computed Target Group Average Price: ${avg_price:.4f} USDT")
    print("Coins currently trading below this benchmark average:")
    
    for coin in data:
        current_price = float(coin.get("lastPrice", 0.0))
        if current_price < avg_price:
            print(f"  - {coin.get('symbol')}: ${current_price:,.4f} USDT")
    print()

# Task 4: Sort coins by traded volume and display the top 5
def rank_coins_by_volume(data):
    # Sort on quoteVolume (total value traded in USDT) in descending order
    sorted_by_volume = sorted(data, key=lambda x: float(x.get("quoteVolume", 0.0)), reverse=True)
    
    print("--- Top 5 Ranked Cryptocurrencies by Traded Volume ---")
    for rank, coin in enumerate(sorted_by_volume[:5], start=1):
        volume = float(coin.get("quoteVolume", 0.0))
        print(f"Rank {rank}: {coin.get('symbol'):<10} | Traded Volume: {volume:,.2f} USDT")
    print()

# Helper workflow coordinator wrapper function
def run_analytics_dashboard(data):
    print("=========================================================")
    print("             REAL-TIME DATA ANALYTICS CORNER             ")
    print("=========================================================\n")
    
    # Task 2 Execution
    volatile_coin, volatility_val = find_most_volatile_coin(data)
    print(f"Most Volatile Market Token : {volatile_coin} ({volatility_val:+.2f}% Swing)\n")
    
    # Task 3 Execution
    display_coins_below_average(data)
    
    # Task 4 Execution
    rank_coins_by_volume(data)
    print("=========================================================")

# Run the pipeline
fetch_and_analyze_crypto()

Successfully archived real-time data array matrix to 'crypto_data.json'.

             REAL-TIME DATA ANALYTICS CORNER             

Most Volatile Market Token : SOLUSDT (+4.20% Swing)

Computed Target Group Average Price: $6353.9805 USDT
Coins currently trading below this benchmark average:
  - ETHUSDT: $1,639.0300 USDT
  - BNBUSDT: $587.7300 USDT
  - ADAUSDT: $0.1635 USDT
  - XRPUSDT: $1.1311 USDT
  - LINKUSDT: $7.7420 USDT
  - DOGEUSDT: $0.0843 USDT
  - SOLUSDT: $64.5100 USDT
  - DOTUSDT: $0.9460 USDT
  - AVAXUSDT: $6.5480 USDT

--- Top 5 Ranked Cryptocurrencies by Traded Volume ---
Rank 1: BTCUSDT    | Traded Volume: 1,326,930,974.24 USDT
Rank 2: ETHUSDT    | Traded Volume: 667,579,741.28 USDT
Rank 3: SOLUSDT    | Traded Volume: 198,813,434.59 USDT
Rank 4: XRPUSDT    | Traded Volume: 125,748,959.76 USDT
Rank 5: BNBUSDT    | Traded Volume: 74,582,974.50 USDT



In [2]:
# Task 5: Automating the job execution hourly with exponential backoff handlers
!pip install schedule

import schedule
import time
import requests

def job_wrapper():
    """Executes code blocks and manages rate-limiting issues dynamically."""
    base_delay = 2
    max_retries = 3
    
    for attempt in range(max_retries):
        print(f"Executing scheduled database sync routine (Attempt {attempt + 1})...")
        status = fetch_and_analyze_crypto()
        
        # If the server drops an HTTP 429 Rate Limit error, apply exponential backoff cooling
        if status == 429:
            retry_delay = base_delay ** attempt
            print(f"Rate limiting active. Pausing processing streams for {retry_delay}s...")
            time.sleep(retry_delay)
        else:
            # Execution passed cleanly; break out of the retry loop
            break

# Configure the schedule engine to trigger the job sequence at regular intervals
schedule.clear()
schedule.every(1).hours.do(job_wrapper)

print("Scheduler configuration complete. Daemon initialized successfully.")

# --- Notebook Simulation Workspace Hook ---
# Running a manual task simulation loop to prove operational viability
print("\n--- Simulating 1 Cron Cycle Loop Step Inside Jupyter Notebook ---")
job_wrapper()

Scheduler configuration complete. Daemon initialized successfully.

--- Simulating 1 Cron Cycle Loop Step Inside Jupyter Notebook ---
Executing scheduled database sync routine (Attempt 1)...
Successfully archived real-time data array matrix to 'crypto_data.json'.

             REAL-TIME DATA ANALYTICS CORNER             

Most Volatile Market Token : BTCUSDT (+4.15% Swing)

Computed Target Group Average Price: $6354.6301 USDT
Coins currently trading below this benchmark average:
  - ETHUSDT: $1,639.7400 USDT
  - BNBUSDT: $587.8300 USDT
  - ADAUSDT: $0.1635 USDT
  - XRPUSDT: $1.1316 USDT
  - LINKUSDT: $7.7450 USDT
  - DOGEUSDT: $0.0844 USDT
  - SOLUSDT: $64.5300 USDT
  - DOTUSDT: $0.9460 USDT
  - AVAXUSDT: $6.5510 USDT

--- Top 5 Ranked Cryptocurrencies by Traded Volume ---
Rank 1: BTCUSDT    | Traded Volume: 1,326,994,609.67 USDT
Rank 2: ETHUSDT    | Traded Volume: 667,351,895.57 USDT
Rank 3: SOLUSDT    | Traded Volume: 198,799,120.80 USDT
Rank 4: XRPUSDT    | Traded Volume: 125,780,73